# 02. От анализа к действию: подключаем Jenkins

## Что агент пока не умеет

После `01` агент умеет исследовать локальный workspace, но не может взаимодействовать с внешними инженерными системами.

> Tool — это контролируемый контракт между рассуждением модели и обычным детерминированным кодом.


## Разделение ответственности

```text
Модель:
- понимает намерение пользователя;
- выбирает действие;
- формирует параметры;
- интерпретирует результат.

Python connector:
- выполняет HTTP-запрос;
- знает authentication;
- проверяет входные данные;
- нормализует ответ;
- обрабатывает ошибки.
```

Модель не видит весь исходный код connector. Она видит описание способности и JSON-схему входа.


## Jenkins capabilities

```text
read:
- job information;
- last build;
- build status;
- build log.

write:
- trigger build.
```

Даже если interrupt не нужен, разделение read/write важно для архитектуры.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

from deepagents import create_deep_agent
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / '.env')

DEFAULT_MODEL = 'openrouter:tencent/hy3:free'

def model_name() -> str:
    return os.getenv('OPENCLAW_MODEL', DEFAULT_MODEL)

def write_text(path: str, content: str) -> Path:
    target = REPO_ROOT / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return target

def write_json(path: str, payload: dict) -> Path:
    target = REPO_ROOT / path
    target.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n')
    return target


In [ ]:
from connectors.jenkins import get_jenkins_job_info, trigger_jenkins_job

print(get_jenkins_job_info.name)
print(get_jenkins_job_info.description)
print(trigger_jenkins_job.args_schema.model_json_schema())


In [ ]:
print(trigger_jenkins_job.invoke({
    "parameters": {"OPENCLAW_SMOKE": "true"},
    "dry_run": True,
}))


In [ ]:
ENTRYPOINT = "\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\nfrom deepagents import create_deep_agent\nfrom dotenv import load_dotenv\nfrom connectors.jenkins import JENKINS_TOOLS\n\nload_dotenv()\nDEFAULT_MODEL = \"openrouter:tencent/hy3:free\"\n\n\ndef _backend():\n    from deepagents.backends import FilesystemBackend\n    return FilesystemBackend(root_dir=Path(os.getenv(\"OPENCLAW_WORKSPACE\", \".\")).resolve(), virtual_mode=True)\n\nBASE_PROMPT = \"\"\"\\\nYou are OpenClaw at stage 02 with Jenkins tools. Separate read actions from write actions.\nUse dry-run before triggering builds unless the user explicitly asks for a real build.\nReturn normalized operational summaries, not raw dumps.\n\"\"\"\n\nagent = create_deep_agent(\n    model=os.getenv(\"OPENCLAW_MODEL\", DEFAULT_MODEL),\n    tools=JENKINS_TOOLS,\n    system_prompt=BASE_PROMPT,\n    backend=_backend(),\n)\n"
CONFIG = {"dependencies": ["."], "graphs": {"openclaw_02": "./agents/openclaw_02_jenkins_tools.py:agent"}, "env": ".env"}
print(write_text('agents/openclaw_02_jenkins_tools.py', ENTRYPOINT).relative_to(REPO_ROOT))
print(write_json('langgraph.openclaw_path.json', CONFIG).relative_to(REPO_ROOT))


## Проверка в LangGraph Studio

### Запрос

```text
Проверь состояние Jenkins job. Скажи результат последней сборки и время её запуска. Затем подготовь smoke build с OPENCLAW_SMOKE=true в dry-run.
```

### Ожидаемое поведение

1. Агент использует read tool для job metadata.
2. Агент не запускает build без явного разрешения.
3. Dry-run показывает method, endpoint, params и credentials mask.

### Что изменилось относительно предыдущего этапа

Появилось внешнее действие через контролируемый Python tool contract.

### Текущее ограничение

Человек всё ещё обязан находиться в LangGraph Studio; агент не имеет привычного пользовательского канала.
